Tesla Deliveries ML Pipeline   
End-to-end: EDA -> preprocessing -> feature engineering ->
regression modeling -> hyperparameter tuning -> forecasting

In [1]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.inspection import permutation_importance
import joblib

In [2]:
# Optional models
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    import statsmodels.api as sm
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATS_MODELS_AVAILABLE = True
except Exception:
    STATS_MODELS_AVAILABLE = False

In [3]:
# ---------------------------
# 1. CONFIG
# ---------------------------
FILE_PATH = "tesla_deliveries_dataset_2015_2025.csv"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

In [4]:
# ---------------------------
# 2. LOAD DATA
# ---------------------------
df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")

print("\nDataset shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nPreview:\n", df.head())


Dataset shape: (2640, 12)

Columns:
 ['Year', 'Month', 'Region', 'Model', 'Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Source_Type', 'Charging_Stations']

Preview:
    Year  Month         Region    Model  Estimated_Deliveries  \
0  2023      5         Europe  Model S                 17646   
1  2015      2           Asia  Model X                  3797   
2  2019      1  North America  Model X                  8411   
3  2021      2  North America  Model 3                  6555   
4  2016     12    Middle East  Model Y                 12374   

   Production_Units  Avg_Price_USD  Battery_Capacity_kWh  Range_km  \
0             17922       92874.27                   120       704   
1              4164       62205.65                    75       438   
2              9189      117887.32                    82       480   
3              7311       89294.91                   120       712   
4             13537      114

In [5]:
# ---------------------------
# 3. BASIC CLEANING
# ---------------------------
df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]

# remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# strip text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [6]:

# ---------------------------
# 4. AUTO-DETECT DATE COLUMN
# ---------------------------
date_candidates = [c for c in df.columns if any(k in c for k in ["date", "month", "quarter", "year", "time"])]
date_col = None

for c in date_candidates:
    try:
        parsed = pd.to_datetime(df[c], errors="coerce")
        if parsed.notna().sum() > max(0.6 * len(df), 3):
            date_col = c
            df[c] = parsed
            break
    except Exception:
        pass

if date_col is None:
    # try the first object-like column as date if it looks like dates
    for c in df.columns:
        if df[c].dtype == "object":
            parsed = pd.to_datetime(df[c], errors="coerce")
            if parsed.notna().sum() > max(0.6 * len(df), 3):
                date_col = c
                df[c] = parsed
                break

print("\nDetected date column:", date_col)


Detected date column: year
